# C5 — Calibração e Escolha de Limiar

## Objetivo

Calibrar os finalistas da shortlist e escolher o limiar operacional usando **somente treino e validação**.

Este notebook:

- carrega o artefato supervisionado canônico via `load_supervised_artifact`;
- reutiliza exatamente a partição persistida — **não cria novo split**;
- treina cada finalista no treino e calibra na validação;
- compara Brier antes/depois para cada método candidato em `calibracao.metodos_candidatos`;
- escolhe o limiar com `escolher_limiar` sobre as probabilidades calibradas na validação;
- salva curva de confiabilidade, decomposição de Brier, ECE, MCE e tabela de limiar;
- **não abre o teste**;
- **não escreve no YAML** — a atualização de `calibracao.metodo`, `limiar.otimo`, `limiar.metrica` e dos gates `G7_calibracao`/`G8_limiar` é decisão do grupo.

> Nota metodológica: este notebook segue `calibracao.fonte: validacao` do `supervised.yaml`. As métricas "depois" refletem o desempenho na mesma validação usada para ajustar o calibrador e, por isso, são otimistas. A avaliação imparcial acontece no teste, uma única vez, via C6.

## 1. Bootstrap — raiz do projeto e imports

In [ ]:
from pathlib import Path
import json
import os
import platform
import sys

import numpy as np
import pandas as pd
from IPython.display import display


def encontrar_raiz_projeto() -> Path:
    atual = Path.cwd().resolve()
    for pasta in [atual, *atual.parents]:
        if (pasta / "src").is_dir() and (pasta / "configs").is_dir():
            return pasta
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto.")


ROOT_DIR = encontrar_raiz_projeto()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

print("ROOT_DIR:", ROOT_DIR)
print("Python:", sys.version.split()[0])
print("Sistema:", platform.system(), platform.release())

In [ ]:
from src.utils import config as C
from src.utils import env
from src.utils.reproducibility import Run
from src.data.supervised_artifact import load_supervised_artifact
from src.evaluation.leakage import check_leakage
from src.evaluation.metrics import metricas_completas, brier
from src.evaluation.calibration import (
    curva_confiabilidade,
    ece,
    mce,
    brier_decomposicao,
)
from src.models.logistic import build_logistic_pipeline
from src.models.boosting import build_boosting_pipeline
from src.models.calibration import calibrar, escolher_limiar

# Checagem amigável de scikit-learn >= 1.6 (FrozenEstimator)
try:
    import sklearn.frozen  # noqa: F401
except ImportError as exc:
    raise RuntimeError(
        "Este notebook requer scikit-learn >= 1.6 "
        "(sklearn.frozen.FrozenEstimator). "
        f"ImportError: {exc}"
    ) from exc

env.carregar_dotenv()
env.garantir_diretorios()

TABLES_DIR = C.TABLES_DIR
PREDICTIONS_DIR = C.PREDICTIONS_DIR
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

print(C.resumo())

## 2. Carregar o artefato supervisionado canônico

Mesma fonte usada no C3 e no C4. O split persistido é reutilizado integralmente.

In [ ]:
COHORT_KIND = os.getenv("SRAG_COHORT_KIND", "provisional").lower()

df, manifest = load_supervised_artifact(
    ROOT_DIR,
    expected_kind=COHORT_KIND,
)

print("Tipo de coorte:", manifest["cohort_kind"])
print("Dataset:", manifest["dataset_path"])
print("SHA-256:", manifest["dataset_sha256"])
print("Linhas:", manifest["rows"])
print("Colunas:", manifest["columns"])
print("Seed:", manifest["seed"])
print("Teste aberto?", manifest["test_opened"])
print("Partições:", manifest["partition_counts"])

display(df.head(3))

## 3. Validar o contrato e verificar gates

- o artefato tem todas as colunas esperadas;
- as três partições estão presentes;
- o teste nunca foi aberto;
- mostramos o estado atual de `G7_calibracao` e `G8_limiar`. O notebook roda independentemente, mas o YAML só deve ser atualizado após aprovação formal.

In [ ]:
PARTICAO = manifest["partition_column"]
ALVO = manifest["target_column"]
FEATURES = list(manifest["feature_columns"])

particoes_observadas = set(
    df[PARTICAO].dropna().astype(str).unique().tolist()
)

alvo_binario = (
    pd.to_numeric(df[ALVO], errors="coerce")
    .dropna()
    .astype(int)
    .isin([0, 1])
    .all()
)

verificacoes = {
    "artefato_nao_vazio": not df.empty,
    "particao_presente": PARTICAO in df.columns,
    "alvo_presente": ALVO in df.columns,
    "features_presentes": set(FEATURES).issubset(df.columns),
    "particoes_esperadas": particoes_observadas == {"treino", "validacao", "teste"},
    "alvo_binario": bool(alvo_binario),
    "teste_fechado_no_manifesto": manifest.get("test_opened") is False,
}

display(pd.DataFrame({
    "verificacao": list(verificacoes.keys()),
    "resultado": list(verificacoes.values()),
}))

if not all(verificacoes.values()):
    falhas = [k for k, v in verificacoes.items() if not v]
    raise RuntimeError(f"Falhas no contrato do artefato: {falhas}")

G7_STATUS = (C.cfg("gates", "G7_calibracao") or {}).get("status")
G8_STATUS = (C.cfg("gates", "G8_limiar") or {}).get("status")

display(pd.DataFrame({
    "gate": ["G7_calibracao", "G8_limiar"],
    "status": [G7_STATUS, G8_STATUS],
}))

print("Contrato validado. Partição persistida reutilizada.")
print("Este notebook NÃO atualiza o YAML automaticamente.")

## 4. Carregar a shortlist de finalistas

Os finalistas vêm de `selecao_modelos.finalistas` do `supervised.yaml`.

Se o C4 tiver produzido um tuning limitado, o grupo deve ter atualizado os hiperparâmetros do finalista no YAML antes de rodar este notebook.

In [ ]:
selecao_cfg = C.cfg("selecao_modelos") or {}
FINALISTAS = list(selecao_cfg.get("finalistas") or [])

if not FINALISTAS:
    raise RuntimeError(
        "`selecao_modelos.finalistas` está vazio no supervised.yaml. "
        "Defina os finalistas antes de rodar o C5."
    )

FINALISTAS_MAP = {
    "logistic": build_logistic_pipeline,
    "hist_gradient_boosting": build_boosting_pipeline,
    "boosting": build_boosting_pipeline,
}

finalistas_desconhecidos = [n for n in FINALISTAS if n not in FINALISTAS_MAP]
if finalistas_desconhecidos:
    raise ValueError(
        f"Finalistas desconhecidos no YAML: {finalistas_desconhecidos}. "
        f"Suportados: {sorted(FINALISTAS_MAP)}."
    )

print("Finalistas:", FINALISTAS)
print("Teste consultado no YAML?", selecao_cfg.get("teste_consultado"))

## 5. Separar treino e validação (sem tocar no teste)

In [ ]:
treino = df.loc[df[PARTICAO] == "treino"].copy()
validacao = df.loc[df[PARTICAO] == "validacao"].copy()

# Não criamos a variável `teste`.

X_train = treino[FEATURES].copy()
y_train = pd.to_numeric(treino[ALVO], errors="coerce").astype("int8")

X_val = validacao[FEATURES].copy()
y_val = pd.to_numeric(validacao[ALVO], errors="coerce").astype("int8")

assert X_train.index.equals(y_train.index)
assert X_val.index.equals(y_val.index)

display(pd.DataFrame({
    "particao": ["treino", "validacao"],
    "n": [len(treino), len(validacao)],
    "positivos": [int(y_train.sum()), int(y_val.sum())],
    "prevalencia": [float(y_train.mean()), float(y_val.mean())],
}))

n_teste = int((df[PARTICAO] == "teste").sum())
print(f"Teste: {n_teste:,} linhas — mantido fechado.")

In [ ]:
check_leakage(X_train)
check_leakage(X_val)

proibidas = set(C.cfg("features", "proibidas") or [])
assert not (set(X_train.columns) & proibidas)
assert not (set(X_val.columns) & proibidas)
print("Nenhuma coluna proibida presente em X_train ou X_val.")

## 6. Treinar cada finalista e calibrar na validação

- O pipeline base é treinado no treino completo.
- Para cada método em `calibracao.metodos_candidatos`, o calibrador é ajustado na **validação** via `calibrar(pipeline, X_val, y_val, method=...)`.
- Comparamos o Brier antes (pipeline bruto) e depois (probabilidades calibradas) na validação.
- O melhor método é o de menor Brier calibrado.

> O viés otimista de "depois" é conhecido e está documentado no cabeçalho. A avaliação imparcial ocorre no C6.

In [ ]:
def construir_finalista(nome: str):
    """Constrói o pipeline base do finalista usando os valores do YAML."""
    if nome == "logistic":
        cfg = C.cfg("models", "logistic") or {}
        return build_logistic_pipeline(
            C=float(cfg.get("C") or 1.0),
            solver=str(cfg.get("solver") or "lbfgs"),
            max_iter=int(cfg.get("max_iter") or 1000),
            class_weight=cfg.get("class_weight") or "balanced",
        )
    if nome in ("hist_gradient_boosting", "boosting"):
        cfg = C.cfg("models", "boosting") or {}
        return build_boosting_pipeline(
            learning_rate=float(cfg.get("learning_rate") or 0.05),
            max_iter=int(cfg.get("max_iter") or 500),
            max_depth=(
                None if cfg.get("max_depth") is None else int(cfg["max_depth"])
            ),
            max_leaf_nodes=int(cfg.get("max_leaf_nodes") or 31),
            min_samples_leaf=int(cfg.get("min_samples_leaf") or 20),
            l2_regularization=float(cfg.get("l2_regularization") or 0.0),
            early_stopping=bool(cfg.get("early_stopping") or False),
            class_weight=cfg.get("class_weight") or "balanced",
        )
    raise ValueError(f"Finalista não suportado: {nome!r}")

In [ ]:
CALIBRACAO_ATIVA = bool((C.cfg("calibracao") or {}).get("ativa", True))
METODOS_CANDIDATOS = list(
    (C.cfg("calibracao") or {}).get("metodos_candidatos") or ["sigmoid", "isotonic"]
)
COMPARAR_BRIER = bool(
    (C.cfg("calibracao") or {}).get("comparar_brier_antes_depois", True)
)

print("Calibração ativa:", CALIBRACAO_ATIVA)
print("Métodos candidatos:", METODOS_CANDIDATOS)
print("Comparar Brier antes/depois:", COMPARAR_BRIER)

resultados_finalistas = {}

for nome in FINALISTAS:
    print(f"\n=== Finalista: {nome} ===")
    pipe_base = construir_finalista(nome)
    pipe_base.fit(X_train, y_train)

    prob_base_val = pipe_base.predict_proba(X_val)[:, 1]
    brier_base = float(brier(y_val, prob_base_val))
    metricas_base = metricas_completas(y_val, prob_base_val, limiar=0.5)

    print(f"Brier (sem calibração): {brier_base:.6f}")
    print(f"AP    (sem calibração): {metricas_base['ap']:.6f}")

    info = {
        "nome": nome,
        "pipeline_base": pipe_base,
        "prob_base_val": prob_base_val,
        "brier_base": brier_base,
        "metricas_base": metricas_base,
        "calibracoes": {},
    }

    if CALIBRACAO_ATIVA:
        for metodo in METODOS_CANDIDATOS:
            try:
                cal = calibrar(pipe_base, X_val, y_val, method=metodo)
                prob_cal = cal.predict_proba(X_val)[:, 1]
                b_cal = float(brier(y_val, prob_cal))
                info["calibracoes"][metodo] = {
                    "pipeline": cal,
                    "prob_val": prob_cal,
                    "brier": b_cal,
                }
                print(f"  - {metodo}: Brier = {b_cal:.6f}")
            except Exception as exc:
                print(f"  - {metodo}: falhou ({exc!r})")

    resultados_finalistas[nome] = info

## 7. Escolher o melhor método de calibração por finalista

Menor Brier na validação vence. Se a calibração estiver desativada ou falhar, mantemos o pipeline bruto.

In [ ]:
escolhas_calibracao = {}

for nome, info in resultados_finalistas.items():
    if not info["calibracoes"]:
        escolhas_calibracao[nome] = {
            "metodo": None,
            "brier": info["brier_base"],
            "ganho_brier": 0.0,
            "sem_calibracao": True,
            "prob_val": info["prob_base_val"],
            "pipeline": info["pipeline_base"],
        }
        continue

    melhor_metodo, melhor_dados = min(
        info["calibracoes"].items(),
        key=lambda kv: kv[1]["brier"],
    )
    ganho = info["brier_base"] - melhor_dados["brier"]
    escolhas_calibracao[nome] = {
        "metodo": melhor_metodo,
        "brier": float(melhor_dados["brier"]),
        "ganho_brier": float(ganho),
        "sem_calibracao": False,
        "prob_val": melhor_dados["prob_val"],
        "pipeline": melhor_dados["pipeline"],
    }

escolhas_df = pd.DataFrame([
    {
        "finalista": nome,
        "metodo_calibracao": info["metodo"],
        "brier_base": resultados_finalistas[nome]["brier_base"],
        "brier_calibrado": info["brier"],
        "ganho_brier": info["ganho_brier"],
    }
    for nome, info in escolhas_calibracao.items()
])

display(escolhas_df)

## 8. Escolher o limiar operacional

Usamos `escolher_limiar` sobre as probabilidades **calibradas** na validação.

- Métrica: `limiar.metrica` do YAML (default `f1`).
- `min_precision`: respeitado quando definido.
- Nenhum valor do teste é consultado.

In [ ]:
limiar_cfg = C.cfg("limiar") or {}
LIMIAR_METRICA = limiar_cfg.get("metrica") or "f1"
LIMIAR_MIN_PRECISION = limiar_cfg.get("min_precision")
LIMIAR_N_LIMIARES = 101

print("Métrica do limiar:", LIMIAR_METRICA)
print("min_precision:", LIMIAR_MIN_PRECISION)

escolhas_limiar = {}

for nome, info in escolhas_calibracao.items():
    prob = info["prob_val"]
    resultado = escolher_limiar(
        y_val,
        prob,
        metrica=LIMIAR_METRICA,
        n_limiares=LIMIAR_N_LIMIARES,
        min_precision=LIMIAR_MIN_PRECISION,
    )
    escolhas_limiar[nome] = {
        "limiar": float(resultado["limiar"]),
        "valor": float(resultado["valor"]),
        "metrica": resultado["metrica"],
        "auc": float(resultado["auc"]),
        "brier": float(resultado["brier"]),
        "tabela": resultado["tabela"],
    }

limiares_df = pd.DataFrame([
    {
        "finalista": nome,
        "limiar": r["limiar"],
        "valor_metrica": r["valor"],
        "metrica": r["metrica"],
        "auc": r["auc"],
        "brier": r["brier"],
    }
    for nome, r in escolhas_limiar.items()
])

display(limiares_df)

## 9. Diagnósticos de calibração

Para cada finalista, com o método escolhido:

- curva de confiabilidade;
- decomposição de Brier (`reliability`, `resolution`, `uncertainty`);
- ECE e MCE.

In [ ]:
diagnosticos = {}

for nome, info in escolhas_calibracao.items():
    prob_cal = info["prob_val"]
    rotulo = info["metodo"] or "sem_calibracao"

    curva = curva_confiabilidade(y_val, prob_cal, n_bins=10)
    decomp = brier_decomposicao(y_val, prob_cal, n_bins=10)
    ece_val = float(ece(y_val, prob_cal, n_bins=10))
    mce_val = float(mce(y_val, prob_cal, n_bins=10))

    diagnosticos[nome] = {
        "rotulo": rotulo,
        "curva": curva,
        "brier_decomp": decomp,
        "ece": ece_val,
        "mce": mce_val,
        "prob_val": prob_cal,
    }

    print(f"\n=== {nome} ({rotulo}) ===")
    print(f"ECE: {ece_val:.6f} | MCE: {mce_val:.6f}")
    print("Decomposição de Brier:")
    for k, v in decomp.items():
        print(f"  {k}: {v:.6f}")
    display(curva)

## 10. Persistir artefatos

In [ ]:
timestamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ")
prefixo = f"C5_calibracao_limiar_{manifest['cohort_kind']}_{timestamp}"

caminho_escolhas_cal = TABLES_DIR / f"{prefixo}_escolhas_calibracao.csv"
caminho_limiares = TABLES_DIR / f"{prefixo}_limiares_escolhidos.csv"
caminho_manifesto = TABLES_DIR / f"{prefixo}_manifesto.json"

escolhas_df.to_csv(caminho_escolhas_cal, index=False, encoding="utf-8-sig")
limiares_df.to_csv(caminho_limiares, index=False, encoding="utf-8-sig")

artefatos_diag = {}
for nome, d in diagnosticos.items():
    c_curva = TABLES_DIR / f"{prefixo}_{nome}_curva_confiabilidade.csv"
    c_limiar = TABLES_DIR / f"{prefixo}_{nome}_tabela_limiar.csv"
    d["curva"].to_csv(c_curva, index=False, encoding="utf-8-sig")
    escolhas_limiar[nome]["tabela"].to_csv(
        c_limiar, index=False, encoding="utf-8-sig"
    )
    artefatos_diag[nome] = {
        "curva_confiabilidade": str(c_curva),
        "tabela_limiar": str(c_limiar),
        "ece": float(d["ece"]),
        "mce": float(d["mce"]),
        "brier_decomposicao": {k: float(v) for k, v in d["brier_decomp"].items()},
        "rotulo_calibracao": d["rotulo"],
    }

predicoes_list = []
for nome, info in resultados_finalistas.items():
    predicoes_list.append(pd.DataFrame({
        "y_true": y_val.to_numpy(),
        "y_prob": info["prob_base_val"],
        "modelo": f"{nome}_base",
        "cohort_kind": manifest["cohort_kind"],
    }))
    metodo = escolhas_calibracao[nome]["metodo"]
    if metodo and metodo in info["calibracoes"]:
        predicoes_list.append(pd.DataFrame({
            "y_true": y_val.to_numpy(),
            "y_prob": info["calibracoes"][metodo]["prob_val"],
            "modelo": f"{nome}_{metodo}",
            "cohort_kind": manifest["cohort_kind"],
        }))

pred_val = pd.concat(predicoes_list, ignore_index=True)
pred_val["y_pred"] = (pred_val["y_prob"] >= 0.5).astype(int)

caminho_predicoes = PREDICTIONS_DIR / f"{prefixo}_predicoes_validacao.parquet"
pred_val.to_parquet(caminho_predicoes, index=False)

features_final_yaml = C.cfg("features", "final")

manifesto_c5 = {
    "cohort_kind": manifest["cohort_kind"],
    "dataset_path": manifest["dataset_path"],
    "dataset_sha256": manifest["dataset_sha256"],
    "feature_columns": FEATURES,
    "target_column": ALVO,
    "partition_column": PARTICAO,
    "n_treino": int(len(treino)),
    "n_validacao": int(len(validacao)),
    "n_teste_fechado": n_teste,
    "finalistas": FINALISTAS,
    "calibracao_ativa": CALIBRACAO_ATIVA,
    "metodos_candidatos": METODOS_CANDIDATOS,
    "escolhas_calibracao": {
        nome: {
            "metodo": info["metodo"],
            "brier_base": float(resultados_finalistas[nome]["brier_base"]),
            "brier_calibrado": float(info["brier"]),
            "ganho_brier": float(info["ganho_brier"]),
        }
        for nome, info in escolhas_calibracao.items()
    },
    "escolhas_limiar": {
        nome: {
            "limiar": float(r["limiar"]),
            "valor_metrica": float(r["valor"]),
            "metrica": r["metrica"],
            "auc": float(r["auc"]),
            "brier": float(r["brier"]),
        }
        for nome, r in escolhas_limiar.items()
    },
    "diagnosticos": artefatos_diag,
    "g7_status_no_momento": G7_STATUS,
    "g8_status_no_momento": G8_STATUS,
    "teste_aberto": False,
    "yaml_atualizado_por_este_notebook": False,
    "features_final_congelada_no_yaml": bool(features_final_yaml),
    "observacoes": [
        "Calibração e limiar escolhidos exclusivamente na validação.",
        "Teste nunca consultado.",
        "Atualização do YAML é decisão do grupo.",
        "Brier 'depois' é otimista: calibrador foi ajustado na mesma validação.",
        "Se features.final ainda for null, finalistas continuam provisórios.",
    ],
    "artefatos": {
        "escolhas_calibracao": str(caminho_escolhas_cal),
        "limiares_escolhidos": str(caminho_limiares),
        "predicoes_validacao": str(caminho_predicoes),
    },
}

caminho_manifesto.write_text(
    json.dumps(manifesto_c5, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

comp_lido = pd.read_csv(caminho_escolhas_cal, encoding="utf-8-sig")
lim_lido = pd.read_csv(caminho_limiares, encoding="utf-8-sig")
pred_lido = pd.read_parquet(caminho_predicoes)
assert len(comp_lido) == len(escolhas_df)
assert len(lim_lido) == len(limiares_df)
assert len(pred_lido) == len(pred_val)

print("Escolhas de calibração:", caminho_escolhas_cal)
print("Limiares escolhidos   :", caminho_limiares)
print("Predições de validação:", caminho_predicoes)
print("Manifesto             :", caminho_manifesto)

## 11. Proposta de atualização do `supervised.yaml`

Este notebook **não escreve no YAML**. As linhas abaixo são uma proposta para revisão do grupo.

Depois de aprovadas:

1. Atualize `calibracao.metodo` e `limiar.otimo` / `limiar.metrica`.
2. Marque `gates.G7_calibracao.status: aprovado` e `gates.G8_limiar.status: aprovado`, com `aprovado_por` e `data`.
3. Rode o C5 novamente se quiser reconferir as escolhas com o YAML já atualizado.
4. Só depois siga para o C6.

In [ ]:
propostas = []

# Escolhe o finalista com maior AP na validação para propor o par calibração/limiar global.
melhor_por_ap = None
metodo_escolhido = None
limiar_escolhido = None

if escolhas_calibracao:
    ap_por_finalista = {
        nome: float(resultados_finalistas[nome]["metricas_base"]["ap"])
        for nome in escolhas_calibracao
    }
    melhor_por_ap = max(ap_por_finalista.items(), key=lambda kv: kv[1])[0]
    metodo_escolhido = escolhas_calibracao[melhor_por_ap]["metodo"]
    limiar_escolhido = escolhas_limiar[melhor_por_ap]

    propostas += [
        {"chave": "calibracao.metodo", "valor_proposto": metodo_escolhido},
        {"chave": "limiar.otimo", "valor_proposto": float(limiar_escolhido["limiar"])},
        {"chave": "limiar.metrica", "valor_proposto": limiar_escolhido["metrica"]},
    ]

for nome, info in escolhas_calibracao.items():
    propostas.append({
        "chave": f"finalista_{nome}.calibracao.metodo",
        "valor_proposto": info["metodo"],
    })
    propostas.append({
        "chave": f"finalista_{nome}.limiar_otimo",
        "valor_proposto": float(escolhas_limiar[nome]["limiar"]),
    })

display(pd.DataFrame(propostas))

print()
if melhor_por_ap is not None:
    print(f"Finalista com maior AP na validação: {melhor_por_ap}")
    print(f"Método de calibração sugerido: {metodo_escolhido}")
    print(f"Limiar sugerido             : {limiar_escolhido['limiar']:.4f} ({limiar_escolhido['metrica']})")
else:
    print("Sem finalistas para propor atualização.")

print()
print("Gates: G7 e G8 permanecem decisão do grupo. Sugestão:")
print(f"  G7_calibracao: status=aprovado (método {metodo_escolhido})")
print(f"  G8_limiar:     status=aprovado (limiar {float(limiar_escolhido['limiar']) if limiar_escolhido else 'n/a'})")
print()
print("Atualize manualmente configs/supervised.yaml e reexecute este notebook se quiser reconferir.")

## 12. Registro da execução

In [ ]:
metricas_run = {}
for nome in FINALISTAS:
    metricas_run[f"{nome}_ap"] = float(resultados_finalistas[nome]["metricas_base"]["ap"])
    metricas_run[f"{nome}_brier_base"] = float(resultados_finalistas[nome]["brier_base"])
    metricas_run[f"{nome}_brier_calibrado"] = float(escolhas_calibracao[nome]["brier"])
    metricas_run[f"{nome}_limiar"] = float(escolhas_limiar[nome]["limiar"])

with Run(
    modelo="C5_calibracao_limiar",
    parametros={
        "cohort_kind": manifest["cohort_kind"],
        "finalistas": FINALISTAS,
        "calibracao_ativa": CALIBRACAO_ATIVA,
        "metodos_candidatos": METODOS_CANDIDATOS,
        "limiar_metrica": LIMIAR_METRICA,
        "min_precision": LIMIAR_MIN_PRECISION,
        "teste_aberto": False,
        "yaml_atualizado": False,
    },
    dataset={
        "n_treino": int(len(treino)),
        "n_validacao": int(len(validacao)),
        "n_teste_fechado": n_teste,
    },
    features={
        "n_features": len(FEATURES),
        "usadas": FEATURES,
    },
    notas=(
        "C5 — calibração e limiar escolhidos na validação. "
        "Teste não consultado. YAML não atualizado automaticamente."
    ),
    dataset_path=Path(manifest["dataset_path"]),
    dataset_sha256=manifest["dataset_sha256"],
) as r:
    r.metrica(**metricas_run)

print("Run registrado em:", r.pasta)

## 13. Conclusão

Este notebook fornece:

- o método de calibração escolhido por finalista (menor Brier na validação);
- o limiar operacional proposto para cada finalista;
- diagnósticos de calibração (curva de confiabilidade, ECE, MCE, decomposição de Brier);
- uma proposta de atualização do `supervised.yaml`.

**Não** atualiza o YAML automaticamente e **não** consulta o teste.

**Próximo passo:** atualizar o YAML com a decisão do grupo, marcar `G7_calibracao` e `G8_limiar` como aprovados e, em seguida, rodar o C6 (validação final) com `G9_holdout` aprovado.

In [ ]:
checklist = {
    "contrato_validado": all(verificacoes.values()),
    "teste_nao_materializado": "teste" not in globals(),
    "finalistas_carregados": len(FINALISTAS) > 0,
    "calibracao_avaliada": all(
        bool(info["calibracoes"]) or not CALIBRACAO_ATIVA
        for info in resultados_finalistas.values()
    ),
    "limiar_escolhido_para_todos": all(
        nome in escolhas_limiar for nome in FINALISTAS
    ),
    "artefatos_salvos": caminho_manifesto.exists(),
    "yaml_nao_modificado_automaticamente": True,
    "g7_status_registrado": True,
    "g8_status_registrado": True,
}

display(pd.DataFrame({
    "item": list(checklist.keys()),
    "concluido": list(checklist.values()),
}))

if not all(checklist.values()):
    pendencias = [k for k, v in checklist.items() if not v]
    raise RuntimeError(f"C5 terminou com pendências: {pendencias}")

print("C5 concluído. Teste fechado, YAML aguardando decisão do grupo.")
print(f"G7 (calibração): {G7_STATUS}")
print(f"G8 (limiar): {G8_STATUS}")
print("Próximo: atualizar supervised.yaml, aprovar G7/G8 e rodar o C6 com G9 aprovado.")